# 第 5 章：无监督预训练

把第 4 章搭好的 GPT-2 模型真正训练起来。本章覆盖：**训练循环 → loss 下降 → 加载 OpenAI 预训练权重 → 生成可读文本**。

> 主线用 `data/the-verdict.txt`（约 2 万字）做 demo 预训练。真实场景需要 TB 级语料 + 数百 GPU。


## 1. 准备数据与模型

用滑动窗口从原文切出 `(input, target)` 训练对，target 是 input 右移一位（下一个 token 预测任务）。

In [ ]:
import torch
from pathlib import Path
import tiktoken
from src.gpt import GPTModel, GPT_CONFIG_124M, create_dataloader_v1

# 读取语料
data_path = Path("data/the-verdict.txt")
with open(data_path, encoding="utf-8") as f:
    text = f.read()
print(f"语料: {len(text):,} 字符")

tok = tiktoken.get_encoding("gpt2")
dataloader = create_dataloader_v1(
    text, batch_size=2, max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"], shuffle=True, drop_last=True,
)
print(f"批次: {len(dataloader)}")

## 2. 训练循环

**下一步预测任务**的 loss：把 logits 和 target 都 flatten 后做交叉熵。这是 GPT 预训练的标准目标。

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    # flatten 成 [batch*seq, vocab] 和 [batch*seq]，对齐做交叉熵
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1), target_batch.flatten()
    )
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total = 0.0
    n = num_batches or len(data_loader)
    for i, (x, y) in enumerate(data_loader):
        if i >= n: break
        total += calc_loss_batch(x, y, model, device).item()
    return total / n


def train_model_simple(model, train_loader, optimizer, device, num_epochs,
                       eval_freq=5):
    """极简训练循环：记录每个 epoch 的训练损失。"""
    model.to(device)
    model.train()
    losses = []
    for epoch in range(num_epochs):
        for x, y in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(x, y, model, device)
            loss.backward()
            optimizer.step()
        # 每个 epoch 记一次平均 loss
        epoch_loss = calc_loss_loader(train_loader, model, device)
        losses.append(epoch_loss)
        print(f"Epoch {epoch+1:2d}/{num_epochs} | loss {epoch_loss:.4f}")
    return losses

## 3. 在小模型上 demo 训练（验证 loss 下降）

完整 124M 在 demo 语料上要跑很久，这里先用**小配置**验证训练循环正确（loss 应下降）。

In [ ]:
torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 小配置：跑得快，只用于验证训练循环
small_cfg = dict(GPT_CONFIG_124M)
small_cfg.update({"emb_dim": 128, "n_layers": 2, "n_heads": 4, "context_length": 256})
small_model = GPTModel(small_cfg)
optimizer = torch.optim.AdamW(small_model.parameters(), lr=4e-4, weight_decay=0.1)

# 重建小上下文的数据加载器
small_dl = create_dataloader_v1(text, batch_size=2, max_length=256, stride=256,
                                shuffle=True, drop_last=True)
losses = train_model_simple(small_model, small_dl, optimizer, device, num_epochs=10)

## 4. 文本生成

训练后用贪婪解码生成文本。未训练模型输出乱码，训练后（哪怕只在 demo 语料上）应出现更像英文的片段。

In [ ]:
from src.gpt import generate_text_simple

def text_to_token_ids(text, tokenizer):
    return torch.tensor([tokenizer.encode(text)])

def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())

start = "I had a little"
start_ids = text_to_token_ids(start, tok).to(device)

small_model.eval()
with torch.no_grad():
    out = generate_text_simple(small_model, start_ids, max_new_tokens=30,
                               context_size=small_cfg["context_length"])
print("生成（小模型，demo 训练后）：")
print(repr(token_ids_to_text(out, tok)))

## 5. 加载 OpenAI 官方权重（关键一步）

demo 训练不足以让 124M 出好文本。原书用 `gpt_download` 脚本下载 OpenAI 官方预训练的 GPT-2 124M 权重，加载到我们的模型后，生成质量大幅提升。

> 加载要点：OpenAI 用的是 `tf.Transpose` 权重布局，需转置后对齐到我们的 `nn.Linear` 参数名。完整加载脚本见 `solution.py` 和原书 5.5 节。

**权重绑定说明**：OpenAI GPT-2 的输出层与 token embedding 共享权重（weight tying），所以原版参数量是 124M；我们的实现未绑定，所以是 163M。ch05 会讲这个。

---
> **本章小结**：训练循环 + 下一步预测 loss + 加载预训练权重 + 生成。
> 完整含 OpenAI 权重加载、loss 曲线绘制的可运行版本见 `solution.py`。